In [24]:
import os
import glob
import pickle
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency


# ============================================================
# Paths
# ============================================================

input_dir = (
    "/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/preprocessing/pkl_files_filtered_with_celltype/"
)

output_dir = os.path.join( "conditional_MI_diagnostics")
os.makedirs(output_dir, exist_ok=True)


In [25]:
pwd()

'/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/analysis/cell_type_analysis'

In [26]:

# ============================================================
# Helper functions
# ============================================================

def cramers_v_corrected(x, y):
    """
    Bias-corrected Cramér's V according to Bergsma (2013).

    x and y are categorical variables.
    """

    confusion_matrix = pd.crosstab(x, y)

    # Need at least 2 categories in both variables
    if confusion_matrix.shape[0] < 2 or confusion_matrix.shape[1] < 2:
        return np.nan

    chi2 = chi2_contingency(
        confusion_matrix,
        correction=False
    )[0]

    n = confusion_matrix.to_numpy().sum()

    if n <= 1:
        return np.nan

    phi2 = chi2 / n

    r, k = confusion_matrix.shape

    # Bias correction
    phi2_corr = max(
        0,
        phi2 - ((k - 1) * (r - 1)) / (n - 1)
    )

    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)

    denominator = min(k_corr - 1, r_corr - 1)

    if denominator <= 0:
        return np.nan

    return np.sqrt(phi2_corr / denominator)


def entropy_from_counts(counts):
    """
    Shannon entropy in bits.
    """
    counts = np.asarray(counts)
    counts = counts[counts > 0]

    if len(counts) == 0:
        return np.nan

    p = counts / counts.sum()

    return -(p * np.log2(p)).sum()



In [27]:

# ============================================================
# Containers
# ============================================================

dataset_rows = []
celltype_rows = []
lineage_rows = []


# ============================================================
# Find pickle files
# ============================================================

pkl_files = sorted(glob.glob(os.path.join(input_dir, "*.pkl")))

print(f"Found {len(pkl_files)} datasets.\n")


Found 7 datasets.



In [28]:

# ============================================================
# Containers
# ============================================================

dataset_rows = []
celltype_rows = []
lineage_rows = []


# ============================================================
# Find pickle files
# ============================================================

pkl_files = sorted(glob.glob(os.path.join(input_dir, "*.pkl")))

print(f"Found {len(pkl_files)} datasets.\n")


# ============================================================
# Analyze datasets
# ============================================================

for file_path in pkl_files:

    dataset_name = os.path.splitext(
        os.path.basename(file_path)
    )[0]

    print("=" * 70)
    print(dataset_name)

    # --------------------------------------------------------
    # Load dataframe
    # --------------------------------------------------------

    with open(file_path, "rb") as f:
        df = pickle.load(f)

    if not isinstance(df, pd.DataFrame):
        print(
            f"WARNING: {dataset_name} is {type(df)}, "
            "not a pandas DataFrame. Skipping."
        )
        continue

    # By your convention:
    celltype_col = df.columns[-2]
    lineage_col = df.columns[-1]

    print(f"Cell type column: {celltype_col}")
    print(f"Lineage column:   {lineage_col}")

    # Work only with cells that have both annotations
    meta = df[[celltype_col, lineage_col]].copy()

    n_before = len(meta)

    meta = meta.dropna(
        subset=[celltype_col, lineage_col]
    )

    n_after = len(meta)

    if n_before != n_after:
        print(
            f"Dropped {n_before - n_after} cells with missing "
            "cell-type or lineage annotation."
        )

    # Cast to string simply to avoid category/object inconsistencies
    meta[celltype_col] = meta[celltype_col].astype(str)
    meta[lineage_col] = meta[lineage_col].astype(str)

    # --------------------------------------------------------
    # Basic counts
    # --------------------------------------------------------

    n_cells = len(meta)
    n_lineages = meta[lineage_col].nunique()
    n_celltypes = meta[celltype_col].nunique()

    lineage_sizes = meta[lineage_col].value_counts()
    celltype_sizes = meta[celltype_col].value_counts()

    # --------------------------------------------------------
    # Cramér's V
    # --------------------------------------------------------

    cramers_v = cramers_v_corrected(
        meta[lineage_col],
        meta[celltype_col]
    )

    # --------------------------------------------------------
    # Dataset-level summary
    # --------------------------------------------------------

    dataset_rows.append({

        "dataset": dataset_name,

        "n_cells": n_cells,
        "n_lineages": n_lineages,
        "n_cell_types": n_celltypes,

        "mean_cells_per_lineage": lineage_sizes.mean(),
        "median_cells_per_lineage": lineage_sizes.median(),
        "min_cells_per_lineage": lineage_sizes.min(),
        "max_cells_per_lineage": lineage_sizes.max(),

        "lineages_ge_2_cells": (lineage_sizes >= 2).sum(),
        "lineages_ge_3_cells": (lineage_sizes >= 3).sum(),
        "lineages_ge_5_cells": (lineage_sizes >= 5).sum(),
        "lineages_ge_10_cells": (lineage_sizes >= 10).sum(),

        "mean_cells_per_celltype": celltype_sizes.mean(),
        "median_cells_per_celltype": celltype_sizes.median(),
        "min_cells_per_celltype": celltype_sizes.min(),
        "max_cells_per_celltype": celltype_sizes.max(),

        "cramers_V_lineage_celltype": cramers_v
    })

    # ========================================================
    # Cell-type-level summaries
    # ========================================================

    for celltype, sub in meta.groupby(celltype_col):

        sizes = sub[lineage_col].value_counts()

        n_ct_cells = len(sub)
        n_ct_lineages = sizes.size

        # Lineage entropy within this cell type:
        # H(lineage | cell type = c)
        lineage_entropy = entropy_from_counts(
            sizes.values
        )

        celltype_rows.append({

            "dataset": dataset_name,
            "cell_type": celltype,

            "n_cells": n_ct_cells,
            "fraction_of_dataset": n_ct_cells / n_cells,

            "n_lineages": n_ct_lineages,

            "mean_cells_per_lineage": sizes.mean(),
            "median_cells_per_lineage": sizes.median(),
            "min_cells_per_lineage": sizes.min(),
            "max_cells_per_lineage": sizes.max(),

            "lineages_ge_2_cells": (sizes >= 2).sum(),
            "lineages_ge_3_cells": (sizes >= 3).sum(),
            "lineages_ge_5_cells": (sizes >= 5).sum(),
            "lineages_ge_10_cells": (sizes >= 10).sum(),

            "fraction_lineages_ge_2_cells":
                (sizes >= 2).mean(),

            "fraction_lineages_ge_3_cells":
                (sizes >= 3).mean(),

            "fraction_lineages_ge_5_cells":
                (sizes >= 5).mean(),

            "fraction_lineages_ge_10_cells":
                (sizes >= 10).mean(),

            "lineage_entropy_bits": lineage_entropy
        })

    # ========================================================
    # Lineage-level summaries
    # ========================================================

    for lineage, sub in meta.groupby(lineage_col):

        ct_counts = sub[celltype_col].value_counts()

        lineage_size = len(sub)
        n_ct = len(ct_counts)

        dominant_fraction = (
            ct_counts.iloc[0] / lineage_size
        )

        celltype_entropy = entropy_from_counts(
            ct_counts.values
        )

        lineage_rows.append({

            "dataset": dataset_name,
            "lineage": lineage,

            "lineage_size": lineage_size,

            "n_cell_types": n_ct,

            "dominant_cell_type": ct_counts.index[0],

            "dominant_cell_type_fraction":
                dominant_fraction,

            "cell_type_entropy_bits":
                celltype_entropy
        })

    print(
        f"{n_cells:,} cells | "
        f"{n_lineages:,} lineages | "
        f"{n_celltypes:,} cell types | "
        f"Cramér's V = {cramers_v:.4f}"
    )


# ============================================================
# Convert to DataFrames
# ============================================================

dataset_summary = pd.DataFrame(dataset_rows)
celltype_summary = pd.DataFrame(celltype_rows)
lineage_summary = pd.DataFrame(lineage_rows)


# ============================================================
# Sort for easier inspection
# ============================================================

dataset_summary = dataset_summary.sort_values(
    "dataset"
)

celltype_summary = celltype_summary.sort_values(
    ["dataset", "n_cells"],
    ascending=[True, False]
)

lineage_summary = lineage_summary.sort_values(
    ["dataset", "lineage_size"],
    ascending=[True, False]
)


# ============================================================
# Save
# ============================================================

dataset_summary.to_csv(
    os.path.join(
        output_dir,
        "dataset_summary.csv"
    ),
    index=False
)

celltype_summary.to_csv(
    os.path.join(
        output_dir,
        "celltype_summary.csv"
    ),
    index=False
)

lineage_summary.to_csv(
    os.path.join(
        output_dir,
        "lineage_summary.csv"
    ),
    index=False
)

Found 7 datasets.

junker_z5_all_genes
Cell type column: cell_type
Lineage column:   barcode_cont
6,191 cells | 210 lineages | 33 cell types | Cramér's V = 0.1898
treutlein_all_genes
Cell type column: cell_type
Lineage column:   barcode_cont
16,506 cells | 3,905 lineages | 13 cell types | Cramér's V = 0.4397
weinreb_all_genes
Cell type column: cell_type
Lineage column:   barcode_cont
49,302 cells | 5,864 lineages | 11 cell types | Cramér's V = 0.6187
you_stage1_barcode_all_genes
Cell type column: cell_type
Lineage column:   barcode_cont
18,650 cells | 17,075 lineages | 6 cell types | Cramér's V = 0.2240
you_stage2_barcode_all_genes
Cell type column: cell_type
Lineage column:   barcode_cont
42,640 cells | 33,308 lineages | 9 cell types | Cramér's V = 0.2740
you_stage3_barcode_all_genes
Cell type column: cell_type
Lineage column:   barcode_cont
23,883 cells | 15,792 lineages | 7 cell types | Cramér's V = 0.3564
you_stage4_barcode_all_genes
Cell type column: cell_type
Lineage column:   ba

In [16]:
dataset_summary

,dataset,n_cells,n_lineages,n_cell_types,mean_cells_per_lineage,median_cells_per_lineage,min_cells_per_lineage,max_cells_per_lineage,lineages_ge_2_cells,lineages_ge_3_cells,lineages_ge_5_cells,lineages_ge_10_cells,mean_cells_per_celltype,median_cells_per_celltype,min_cells_per_celltype,max_cells_per_celltype,cramers_V_lineage_celltype
0,junker_z5_all_genes,6191,210,33,29.480952,7.0,1,458,196,180,143,85,187.606061,94.0,2,945,0.189808
1,treutlein_all_genes,16506,3905,13,4.226889,1.0,1,801,1216,796,449,235,1269.692308,736.0,20,4426,0.439699
2,weinreb_all_genes,49302,5864,11,8.407572,4.0,2,177,5864,4266,2817,1437,4482.000000,1035.0,49,23770,0.618657
3,you_stage1_barcode_all_genes,18650,17075,6,1.092240,1.0,1,10,1220,258,19,1,3108.333333,3548.0,679,4293,0.224018
4,you_stage2_barcode_all_genes,42640,33308,9,1.280173,1.0,1,83,3598,1818,697,129,4737.777778,4856.0,106,14878,0.274012
5,you_stage3_barcode_all_genes,23883,15792,7,1.512348,1.0,1,191,2531,1425,647,169,3411.857143,1827.0,594,10991,0.356361
6,you_stage4_barcode_all_genes,2864,2004,10,1.429142,1.0,1,112,326,132,48,11,286.400000,212.0,27,600,0.387327


In [22]:
celltype_summary[
    [
        "dataset",
        "cell_type",
        "n_cells",
        "fraction_of_dataset",
        "n_lineages",
        "mean_cells_per_lineage",
        "median_cells_per_lineage",
        "lineages_ge_2_cells",
        "lineages_ge_3_cells",
        "lineages_ge_5_cells",
        "lineages_ge_10_cells",
        "lineage_entropy_bits"
    ]
].round(3)

,dataset,cell_type,n_cells,fraction_of_dataset,n_lineages,mean_cells_per_lineage,median_cells_per_lineage,lineages_ge_2_cells,lineages_ge_3_cells,lineages_ge_5_cells,lineages_ge_10_cells,lineage_entropy_bits
2,junker_z5_all_genes,epidermal_cells,945,0.153,84,11.250,3.0,53,48,33,19,4.934
15,junker_z5_all_genes,mesodermal_origin,933,0.151,132,7.068,2.0,88,61,44,22,5.785
10,junker_z5_all_genes,keratinocytes,575,0.093,93,6.183,2.0,51,36,28,15,5.198
18,junker_z5_all_genes,neuronal_precursors,486,0.079,90,5.400,2.0,57,39,24,15,5.400
19,junker_z5_all_genes,neurons,335,0.054,89,3.764,1.0,43,31,18,8,5.510
...,...,...,...,...,...,...,...,...,...,...,...,...
88,you_stage4_barcode_all_genes,IV-mGluts,159,0.056,152,1.046,1.0,7,0,0,0,7.225
80,you_stage4_barcode_all_genes,IV-Cycling,126,0.044,45,2.800,1.0,3,2,2,2,3.194
87,you_stage4_barcode_all_genes,IV-mDAs,121,0.042,115,1.052,1.0,6,0,0,0,6.820
81,you_stage4_barcode_all_genes,IV-GABAs,99,0.035,95,1.042,1.0,4,0,0,0,6.549


In [21]:
dataset = "weinreb_all_genes"

lineage_summary[
    lineage_summary["dataset"] == dataset
][
    [
        "lineage",
        "lineage_size",
        "n_cell_types",
        "dominant_cell_type",
        "dominant_cell_type_fraction",
        "cell_type_entropy_bits"
    ]
].sort_values(
    "lineage_size",
    ascending=False
).round(3)

,lineage,lineage_size,n_cell_types,dominant_cell_type,dominant_cell_type_fraction,cell_type_entropy_bits
4574,1410,177,4,Neutrophil,0.638,1.341
9116,55,165,4,Monocyte,0.382,1.593
4583,1419,142,3,Baso,0.901,0.501
4540,1380,134,5,Monocyte,0.672,1.310
4502,1346,130,5,Neutrophil,0.554,1.465
...,...,...,...,...,...,...
5734,2455,2,1,Undifferentiated,1.000,-0.000
5731,2452,2,2,Undifferentiated,0.500,1.000
5728,245,2,1,Baso,1.000,-0.000
5725,2447,2,1,Undifferentiated,1.000,-0.000


In [29]:
normal_dir = (
    "/fast/AG_Metzger/philipp/python_files/gene_signature/"
    "mutual_information_approach/November_25_ReDo/"
    "calc_files/MI/"
    "mi_analysis_results_MI_label_entropy_all_genes_parallel_with_BH"
)

conditional_dir = (
    "/fast/AG_Metzger/philipp/python_files/gene_signature/"
    "mutual_information_approach/November_25_ReDo/"
    "calc_files/MI/"
    "conditional_mi_analysis_results_MI_label_entropy_all_genes_parallel_with_BH"
)